In [8]:
import os
from os import listdir
import pandas as pd
import numpy as np
import glob
import cv2
import json


from tqdm import tqdm

import torch 
from torchvision import models
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn 
import torchvision.transforms as transforms
from torchsummary import summary

from torch.nn.modules.activation import Sigmoid
from torch.nn.modules.pooling import AdaptiveAvgPool2d

from sklearn.model_selection import train_test_split


import matplotlib.pyplot as plt 
from PIL import Image


from datetime import datetime

from os.path import expanduser

In [9]:
from imgaug.augmentables.kps import KeypointsOnImage
from imgaug.augmentables.kps import Keypoint
import imgaug.augmenters as iaa


In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [11]:
# getting the optimizer and loss_function 

def get_essentials():
    loss_fun = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    return loss_fun, optimizer

# defining batch_train and accuracy functions

def train_batch(data, model, loss_fun, optimizer):
    model.train()
    img, true_points = data
    pred_points = model(img)
    loss_val = loss_fun(pred_points, true_points)
    loss_val.backward()
    optimizer.step()
    optimizer.zero_grad()
    return loss_val.item()

@torch.no_grad()
def val_batch(data, model, loss_fun, optimizer):
    model.eval()
    img, true_points = data
    pred_points = model(img)
    loss_val = loss_fun(pred_points, true_points)
    return loss_val.item()

In [12]:
class KpDataset(Dataset):
    def __init__(self, df):
        super().__init__()
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                              std=[0.229, 0.224, 0.225])
        self.df = df

    def __getitem__(self, ix):
        doc = self.df.iloc[ix]
        image_name = doc[1]
        image_path = os.path.join(root_dir,image_name)  
#         print("image_path",image_path)
        img_arr = cv2.imread(image_path)
        x_axis_points = doc[2:][::2]
        y_axis_points = doc[3:][::2]
        x_axis_points = list(x_axis_points/img_arr.shape[1])           # scaling the facial points with respective image_dim 
        y_axis_points = list(y_axis_points/img_arr.shape[0])           # so that we can use relative positions w.r.t image

        points = x_axis_points + y_axis_points  

        img_arr = img_arr/255.0                                                                 # scaling the image 
        img = self.preprocess_img(img_arr)

        return img.to(device), torch.tensor(points).to(device)

    def __len__(self):
        return self.df.shape[0]


    def preprocess_img(self, img):
        img = cv2.resize(img, (224,224)) 
        img = torch.tensor(img).permute(2,0,1) 
        img = self.normalize(img).float()
        return img

    def load_img(self, ix):
        doc = self.df.iloc[ix]
        img_name = doc[1]
        img = cv2.imread(os.path.join(root_dir, img_name))
        return img

In [13]:

dataset_file = "merge_folder-2022-11-04-13-26"
labes_file = 'new_kps_results.csv'



home = expanduser("~")
parent_path =  home + "/Pictures/" + "Data/"
root_dir = parent_path + dataset_file
data = pd.read_csv(parent_path + labes_file )

In [ ]:

# root_dir = '/home/fearless/Pictures/Data/oct25_combined/images'
all_img_paths = glob.glob(os.path.join(root_dir, '*.jpg'))

# data = pd.read_csv('oct25_combined_json.csv')

train_df, rem_df = train_test_split(data, train_size=0.8)
val_df, test_df = train_test_split(rem_df, test_size=0.5)
train_dataset = KpDataset(train_df)
val_dataset = KpDataset(val_df)
test_dataset = KpDataset(test_df)

batch_sizes = [32,16]
epochs_lst = [2,10,50,100]
# model_type = [models.vgg19(pretrained=True)]
train_epoch_list = []
val_epoch_list = []
config_name_list = []

for batch_size in batch_sizes:
    for epochs in epochs_lst:
        train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=True)



        model = models.vgg19(pretrained=True)
        for param in model.parameters():
              param.requires_grad = False

        pool_layer = nn.Sequential(
             nn.Conv2d(512,512, kernel_size=3, padding='same'),
             nn.ReLU(),
             nn.MaxPool2d(kernel_size=2, stride=2),
             nn.Conv2d(512,50, kernel_size=3, padding='same'),
             nn.ReLU(),
             nn.MaxPool2d(kernel_size=2, stride=2),
             nn.AdaptiveAvgPool2d(output_size=(8,8))
        )

        model.avgpool = pool_layer

        final_predictor = nn.Sequential(
            nn.Linear(3200, 300),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(300, 12),
            nn.Sigmoid()
        )

        model.classifier = final_predictor
        model = model.to(device)

        loss_fun, optimizer = get_essentials()


        # training and validation loops 
        train_epoch, val_epoch = [], []
        for epoch in tqdm(range(epochs)):
            train_batch_losses, val_batch_losses = [], []
            for data in train_dataloader:
                train_batch_loss = train_batch(data, model, loss_fun, optimizer)
                train_batch_losses.append(train_batch_loss)
            for data in val_dataloader:
                val_batch_loss = val_batch(data, model, loss_fun, optimizer)
                val_batch_losses.append(val_batch_loss)
            train_epoch.append(np.mean(train_batch_losses))
            val_epoch.append(np.mean(val_batch_losses))
            


        train_epoch_list.append(train_epoch)
        val_epoch_list.append(val_epoch)
        # Specify a path

        current_moment = datetime.now()
        current_time = current_moment.strftime("%d-%m-%Y_%H-%M-%S")
        print("Current Time : ", current_time)

        PATH = f"../models/m_{str(epochs)}_{str(batch_size)}_{current_time}.pth"
        plt.plot(range(epochs), train_epoch, label="train_loss")
        plt.plot(range(epochs), val_epoch, label="val_loss")
        plt.legend()
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.title("Training Franka Keypoints model")
        image_path = f"{PATH}.png"
        plt.savefig(image_path)
        
        config_name_list.append(PATH)

        # Save
        torch.save(model, PATH)

/home/jc-merlab/.local/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and will be removed in 0.15, please use 'weights' instead.
  warnings.warn(
/home/jc-merlab/.local/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and will be removed in 0.15. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|█████████████████████████████████████████████| 2/2 [01:45<00:00, 52.69s/it]


Current Time :  04-11-2022_16-03-09


100%|███████████████████████████████████████████| 10/10 [08:41<00:00, 52.14s/it]


Current Time :  04-11-2022_16-11-52


 70%|██████████████████████████████             | 35/50 [29:51<12:54, 51.64s/it]

In [ ]:
print("epochs_lst[0]",epochs_lst[0])
print("train_epoch_list[0]",train_epoch_list[0])
print("val_epoch_list[0]",val_epoch_list[0])

plt.plot(range(epochs_lst[0]), train_epoch_list[0], label="train_loss")
plt.plot(range(epochs_lst[0]), val_epoch_list[0], label="val_loss")
plt.legend()
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training Franka Keypoints model")
image_path = f"{PATH}.png"
plt.savefig(image_path)
plt.show()
